In [ ]:
import time
from typing import Callable

import onnxruntime
import pandas as pd
import torch
import torch.utils.data
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from esp_ppq.executor.torch import TorchExecutor
from esp_ppq.IR import BaseGraph
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Subset
from tqdm import tqdm


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k
    prec1, prec5 = accuracy(output.data, target, topk=(1, 5))
    """
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


def load_imagenet_from_directory(
    directory: str,
    subset: int = None,
    batchsize: int = 32,
    shuffle: bool = False,
    require_label: bool = True,
    num_of_workers: int = 12,
) -> torch.utils.data.DataLoader:
    """
    A standardized Imagenet data loading process,
    directory: The location where the data is loaded
    subset: If set to a non-empty value, a subset of the specified size is extracted from the dataset
    batchsize: The batch size of the data loader
    require_label: Whether labels are required
    shuffle: Whether to shuffle the dataset
    """
    dataset = datasets.ImageFolder(
        directory,
        transforms.Compose(
            [
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        ),
    )

    if subset:
        dataset = Subset(dataset, indices=[_ for _ in range(0, subset)])
    if require_label:
        return torch.utils.data.DataLoader(
            dataset=dataset,
            batch_size=batchsize,
            shuffle=shuffle,
            num_workers=num_of_workers,
            pin_memory=False,
            drop_last=True,  # onnx 模型不支持动态 batchsize，最后一个批次的数据尺寸可能不对齐，因此丢掉最后一个批次的数据
        )
    else:
        return torch.utils.data.DataLoader(
            dataset=dataset,
            batch_size=batchsize,
            shuffle=shuffle,
            num_workers=num_of_workers,
            pin_memory=False,
            collate_fn=lambda x: torch.cat(
                [sample[0].unsqueeze(0) for sample in x], dim=0
            ),
            drop_last=False,  # 不需要标签的数据为 calib 数据，无需 drop
        )


def evaluate_torch_module_with_imagenet(
    model: torch.nn.Module,
    imagenet_validation_dir: str = None,
    batchsize: int = 32,
    device: str = "cuda",
    imagenet_validation_loader: DataLoader = None,
    verbose: bool = True,
) -> pd.DataFrame:
    model.eval()
    with torch.no_grad():
        model_forward_function = lambda input_tensor: model(input_tensor)
        return _evaluate_any_module_with_imagenet(
            model_forward_function=model_forward_function,
            batchsize=batchsize,
            device=device,
            imagenet_validation_dir=imagenet_validation_dir,
            imagenet_validation_loader=imagenet_validation_loader,
            verbose=verbose,
        )


def evaluate_onnx_module_with_imagenet(
    onnxruntime_model_path: str,
    imagenet_validation_dir: str = None,
    batchsize: int = 32,
    device: str = "cuda",
    imagenet_validation_loader: DataLoader = None,
    verbose: bool = True,
) -> pd.DataFrame:
    sess = onnxruntime.InferenceSession(
        path_or_bytes=onnxruntime_model_path, providers=["CUDAExecutionProvider"]
    )
    input_placeholder_name = sess.get_inputs()[0].name
    with torch.no_grad():
        model_forward_function = lambda input_tensor: torch.tensor(
            sess.run(
                input_feed={input_placeholder_name: input_tensor.cpu().numpy()},
                output_names=None,
            )
        )[0]
        return _evaluate_any_module_with_imagenet(
            model_forward_function=model_forward_function,
            batchsize=batchsize,
            device=device,
            imagenet_validation_dir=imagenet_validation_dir,
            imagenet_validation_loader=imagenet_validation_loader,
            verbose=verbose,
        )


def evaluate_mmlab_module_with_imagenet(
    model: torch.nn.Module,
    imagenet_validation_dir: str = None,
    batchsize: int = 32,
    device: str = "cuda",
    imagenet_validation_loader: DataLoader = None,
    verbose: bool = True,
) -> pd.DataFrame:
    model.eval()
    with torch.no_grad():
        model_forward_function = lambda input_tensor: model.forward_test(
            input_tensor, img_metas={}
        )
        return _evaluate_any_module_with_imagenet(
            model_forward_function=model_forward_function,
            batchsize=batchsize,
            device=device,
            imagenet_validation_dir=imagenet_validation_dir,
            imagenet_validation_loader=imagenet_validation_loader,
            verbose=verbose,
        )


def evaluate_ppq_module_with_imagenet(
    model: BaseGraph,
    imagenet_validation_dir: str = None,
    batchsize: int = 32,
    device: str = "cuda",
    imagenet_validation_loader: DataLoader = None,
    verbose: bool = True,
) -> pd.DataFrame:

    executor = TorchExecutor(graph=model, device=device)
    model_forward_function = lambda input_tensor: torch.tensor(
        executor(*[input_tensor])[0]
    )
    return _evaluate_any_module_with_imagenet(
        model_forward_function=model_forward_function,
        batchsize=batchsize,
        device=device,
        imagenet_validation_dir=imagenet_validation_dir,
        imagenet_validation_loader=imagenet_validation_loader,
        verbose=verbose,
    )


def _evaluate_any_module_with_imagenet(
    model_forward_function: Callable,
    imagenet_validation_dir: str,
    batchsize: int = 32,
    device: str = "cuda",
    imagenet_validation_loader: DataLoader = None,
    verbose: bool = True,
):
    """
    一套十分标准的imagenet测试逻辑
    """

    recorder = {"loss": [], "top1_accuracy": [], "top5_accuracy": [], "batch_time": []}

    if imagenet_validation_loader is None:
        imagenet_validation_loader = load_imagenet_from_directory(
            imagenet_validation_dir, batchsize=batchsize, shuffle=False
        )

    loss_fn = torch.nn.CrossEntropyLoss().to("cpu")

    for batch_idx, (batch_input, batch_label) in tqdm(
        enumerate(imagenet_validation_loader),
        desc="Evaluating Model...",
        total=len(imagenet_validation_loader),
    ):
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)
        batch_time_mark_point = time.time()

        batch_pred = model_forward_function(batch_input)
        if isinstance(batch_pred, list):
            batch_pred = torch.tensor(batch_pred)

        recorder["batch_time"].append(time.time() - batch_time_mark_point)
        recorder["loss"].append(loss_fn(batch_pred.to("cpu"), batch_label.to("cpu")))
        prec1, prec5 = accuracy(
            torch.tensor(batch_pred).to("cpu"), batch_label.to("cpu"), topk=(1, 5)
        )
        recorder["top1_accuracy"].append(prec1.item())
        recorder["top5_accuracy"].append(prec5.item())

        if batch_idx % 100 == 0 and verbose:
            print(
                "Test: [{0} / {1}]\t"
                "Prec@1 {top1:.3f} ({top1:.3f})\t"
                "Prec@5 {top5:.3f} ({top5:.3f})".format(
                    batch_idx,
                    len(imagenet_validation_loader),
                    top1=sum(recorder["top1_accuracy"])
                    / len(recorder["top1_accuracy"]),
                    top5=sum(recorder["top5_accuracy"])
                    / len(recorder["top5_accuracy"]),
                )
            )

    if verbose:
        print(
            " * Prec@1 {top1:.3f} Prec@5 {top5:.3f}".format(
                top1=sum(recorder["top1_accuracy"]) / len(recorder["top1_accuracy"]),
                top5=sum(recorder["top5_accuracy"]) / len(recorder["top5_accuracy"]),
            )
        )

    # dump records toward dataframe
    dataframe = pd.DataFrame()
    for column_name in recorder:
        dataframe[column_name] = recorder[column_name]
    return dataframe


    ___________ ____        ____  ____  ____
   / ____/ ___// __ \      / __ \/ __ \/ __ \
  / __/  \__ \/ /_/ /_____/ /_/ / /_/ / / / /
 / /___ ___/ / ____/_____/ ____/ ____/ /_/ /
/_____//____/_/         /_/   /_/    \___\_\




In [5]:
import os
import subprocess
from typing import Iterable, List, Tuple

import torch
import torchvision
from esp_ppq import QuantizationSettingFactory, QuantizationSetting
from esp_ppq.api import espdl_quantize_torch, get_target_platform
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data.dataset import Subset
from torchvision.models.mobilenetv2 import MobileNet_V2_Weights
import torch.nn as nn
import urllib.request
import zipfile


def convert_relu6_to_relu(model):
    for child_name, child in model.named_children():
        if isinstance(child, nn.ReLU6):
            setattr(model, child_name, nn.ReLU())
        else:
            convert_relu6_to_relu(child)
    return model


def quant_setting_mobilenet_v2(
    model: nn.Module,
    optim_quant_method: List[str] = None,
) -> Tuple[QuantizationSetting, nn.Module]:
    """Quantize torch model with optim_quant_method.

    Args:
        optim_quant_method (List[str]): support 'MixedPrecision_quantization', 'LayerwiseEqualization_quantization'
        -'MixedPrecision_quantization': if some layers in model have larger errors in 8-bit quantization, dispathching
                                        the layers to 16-bit quantization. You can remove or add layers according to your
                                        needs.
        -'LayerwiseEqualization_quantization'： using weight equalization strategy, which is proposed by Markus Nagel.
                                                Refer to paper https://openaccess.thecvf.com/content_ICCV_2019/papers/Nagel_Data-Free_Quantization_Through_Weight_Equalization_and_Bias_Correction_ICCV_2019_paper.pdf for more information.
                                                Since ReLU6 exists in MobilenetV2, convert ReLU6 to ReLU for better precision.

    Returns:
        [tuple]: [QuantizationSetting, nn.Module]
    """
    quant_setting = QuantizationSettingFactory.espdl_setting()
    if optim_quant_method is not None:
        if "MixedPrecision_quantization" in optim_quant_method:
            # These layers have larger errors in 8-bit quantization, dispatching to 16-bit quantization.
            # You can remove or add layers according to your needs.
            quant_setting.dispatching_table.append(
                "/features/features.1/conv/conv.0/conv.0.0/Conv",
                get_target_platform(TARGET, 16),
            )
            quant_setting.dispatching_table.append(
                "/features/features.1/conv/conv.0/conv.0.2/Clip",
                get_target_platform(TARGET, 16),
            )
        elif "LayerwiseEqualization_quantization" in optim_quant_method:
            # layerwise equalization
            quant_setting.equalization = True
            quant_setting.equalization_setting.iterations = 4
            quant_setting.equalization_setting.value_threshold = 0.4
            quant_setting.equalization_setting.opt_level = 2
            quant_setting.equalization_setting.interested_layers = None
            # replace ReLU6 with ReLU
            model = convert_relu6_to_relu(model)
        else:
            raise ValueError(
                "Please set optim_quant_method correctly. Support 'MixedPrecision_quantization', 'LayerwiseEqualization_quantization'"
            )

    return quant_setting, model


def collate_fn1(x: Tuple) -> torch.Tensor:
    return torch.cat([sample[0].unsqueeze(0) for sample in x], dim=0)


def collate_fn2(batch: torch.Tensor) -> torch.Tensor:
    return batch.to(DEVICE)


def report_hook(blocknum, blocksize, total):
    downloaded = blocknum * blocksize
    percent = downloaded / total * 100
    print(f"\rDownloading calibration dataset: {percent:.2f}%", end="")


if __name__ == "__main__":
    BATCH_SIZE = 32
    INPUT_SHAPE = [3, 224, 224]
    DEVICE = "cpu"  #  'cuda' or 'cpu', if you use cuda, please make sure that cuda is available
    TARGET = "esp32p4"  #  'c', 'esp32s3' or 'esp32p4'
    NUM_OF_BITS = 8
    ESPDL_MODEL_PATH = "models/torch/mobilenet_v2.espdl"
    CALIB_DIR = "./imagenet"

    # Download mobilenet_v2 from torchvision and dataset
    model = torchvision.models.mobilenet.mobilenet_v2(
        weights=MobileNet_V2_Weights.IMAGENET1K_V1
    )
    model = model.to(DEVICE)
    imagenet_url = "https://dl.espressif.com/public/imagenet_calib.zip"
    os.makedirs(CALIB_DIR, exist_ok=True)
    if not os.path.exists("imagenet_calib.zip"):
        urllib.request.urlretrieve(
            imagenet_url, "imagenet_calib.zip", reporthook=report_hook
        )
    if not os.path.exists(os.path.join(CALIB_DIR, "calib")):
        with zipfile.ZipFile("imagenet_calib.zip", "r") as zip_file:
            zip_file.extractall(CALIB_DIR)
    CALIB_DIR = os.path.join(CALIB_DIR, "calib")

    # -------------------------------------------
    # Prepare Calibration Dataset
    # --------------------------------------------
    if os.path.exists(CALIB_DIR):
        print(f"load imagenet calibration dataset from directory: {CALIB_DIR}")
        dataset = datasets.ImageFolder(
            CALIB_DIR,
            transforms.Compose(
                [
                    transforms.Resize(256),
                    transforms.CenterCrop(224),
                    transforms.ToTensor(),
                    transforms.Normalize(
                        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                    ),
                ]
            ),
        )
        dataset = Subset(dataset, indices=[_ for _ in range(0, 1024)])
        dataloader = DataLoader(
            dataset=dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=4,
            pin_memory=False,
            collate_fn=collate_fn1,
        )
    else:
        # Random calibration dataset only for debug
        print("load random calibration dataset")

        def load_random_calibration_dataset() -> Iterable:
            return [torch.rand(size=INPUT_SHAPE) for _ in range(BATCH_SIZE)]

        # Load training data for creating a calibration dataloader.
        dataloader = DataLoader(
            dataset=load_random_calibration_dataset(),
            batch_size=BATCH_SIZE,
            shuffle=False,
        )

    # -------------------------------------------
    # Quantize Torch Model.
    # --------------------------------------------

    # create a setting for quantizing your network with ESPDL.
    # if you don't need to optimize quantization, set the input 1 of the quant_setting_mobilenet_v2 function None
    # Example: Using LayerwiseEqualization_quantization
    quant_setting, model = quant_setting_mobilenet_v2(
        model, ["LayerwiseEqualization_quantization"]
    )

    quant_ppq_graph = espdl_quantize_torch(
        model=model,
        espdl_export_file=ESPDL_MODEL_PATH,
        calib_dataloader=dataloader,
        calib_steps=32,
        input_shape=[1] + INPUT_SHAPE,
        target=TARGET,
        num_of_bits=NUM_OF_BITS,
        collate_fn=collate_fn2,
        setting=quant_setting,
        device=DEVICE,
        error_report=True,
        skip_export=False,
        export_test_values=False,
        verbose=1,
    )

    # -------------------------------------------
    # Evaluate Quantized Model.
    # --------------------------------------------
    evaluate_ppq_module_with_imagenet(
        model=quant_ppq_graph,
        imagenet_validation_dir=CALIB_DIR,
        batchsize=BATCH_SIZE,
        device=DEVICE,
        verbose=1,
    )

[12:22:06] PPQ Layerwise Equalization Pass Running ...  41 equalization pair(s) was found, ready to run optimization.


Layerwise Equalization: 100%|██████████| 4/4 [00:00<00:00, 107.69it/s]

[/features/features.0/features.0.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.1/conv/conv.0/conv.0.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.1/conv/conv.1/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.2/conv/conv.0/conv.0.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.2/conv/conv.1/conv.1.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.2/conv/conv.2/Conv(Type: Conv, Num of Input: 3, Num of Output: 1), /features/features.3/conv/conv.2/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.3/conv/conv.0/conv.0.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.3/conv/conv.1/conv.1.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.4/conv/conv.0/conv.0.0/Conv(Type: Conv, Num of Input: 3, Num of Output: 1)]
[/features/features.4/conv/conv.1/conv.1.0/Conv(Type: Conv, Num of Input: 3,

Finished.
[12:22:07] PPQ Quantize Simplify Pass Running ...         Finished.
[12:22:07] PPQ Parameter Quantization Pass Running ...    Finished.
[12:22:07] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 2): 100%|██████████| 32/32 [00:19<00:00,  1.61it/s]

...

Test: [0 / 125]	Prec@1 81.250 (81.250)	Prec@5 90.625 (90.625)


Evaluating Model...:  81%|████████  | 101/125 [01:30<00:20,  1.16it/s]

Test: [100 / 125]	Prec@1 70.452 (70.452)	Prec@5 89.140 (89.140)


Evaluating Model...: 100%|██████████| 125/125 [01:50<00:00,  1.13it/s]

 * Prec@1 69.525 Prec@5 88.550
